# Training VGG16 - Klasifikasi Model Tas (3 Kelas)
**Proyek Riset Sistem Pakar & Computer Vision**  
Program Studi Informatika - Universitas Udayana

Notebook ini mengimplementasikan metode **Transfer Learning** menggunakan arsitektur **VGG16** untuk klasifikasi kategori **TAS** ke dalam 3 kelas:
- 
- 
- 

---
### Alur Pipeline:
1. **Setup & Device Configuration** (Deteksi otomatis GPU CUDA / Apple Silicon MPS / CPU)
2. **Data Ingestion & Augmentation** (ImageNet 224x224, Random Flip, Rotation, Color Jitter)
3. **VGG-16 Architecture Adaptation** (Pre-trained ImageNet backbone + Custom Dense Classifier Head)
4. **Training with Regularization & Early Stopping** (CrossEntropyLoss, AdamW, ReduceLROnPlateau)
5. **Visualisasi Kurva Training** (Loss vs Epochs, Accuracy vs Epochs)
6. **Evaluasi Model** (Confusion Matrix Heatmap & Classification Report: Precision, Recall, F1-Score)
7. **Inference Demo** (Prediksi citra baru beserta visualisasi tingkat kepercayaan / confidence score)

In [ ]:
import os
import time
import copy
import random
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
import torchvision
from torchvision import transforms, datasets
from torchvision.models import vgg16, VGG16_Weights

from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")

In [ ]:
# Deteksi akselerator komputasi (CUDA / MPS / CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Hardware Acceleration: NVIDIA GPU ({torch.cuda.get_device_name(0)})")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Hardware Acceleration: Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Hardware Acceleration: CPU Mode")

In [ ]:
# Unduh & ekstrak dataset terlabeli secara otomatis jika belum tersedia
DATASET_DIR = "dataset"
CATEGORY_DIR = os.path.join(DATASET_DIR, "tas")
ZIP_URL = "http://202.10.47.34/slides/dataset_fashion_labeled.zip"
LOCAL_ZIP = "dataset_fashion_labeled.zip"

if not os.path.exists(CATEGORY_DIR):
    if not os.path.exists(LOCAL_ZIP):
        print(f"Mengunduh dataset dari {ZIP_URL}...")
        urllib.request.urlretrieve(ZIP_URL, LOCAL_ZIP)
        print("Unduhan selesai!")
    
    print("Mengekstrak berkas dataset...")
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        zf.extractall(DATASET_DIR)
    print("Ekstraksi dataset selesai!")

print(f"Path Dataset Target: {CATEGORY_DIR}")
for cls_name in sorted(os.listdir(CATEGORY_DIR)):
    cls_path = os.path.join(CATEGORY_DIR, cls_name)
    if os.path.isdir(cls_path):
        n_imgs = len([f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        print(f"  - Kelas '{cls_name}': {n_imgs} gambar")

In [ ]:
# Standar preprocessing dan augmentasi untuk VGG16 (Input 224x224, Normalisasi ImageNet)
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Muat dataset
full_dataset = datasets.ImageFolder(root=CATEGORY_DIR, transform=train_transforms)
class_names = full_dataset.classes
print(f"Kelas terdeteksi ({len(class_names)}): {class_names}")

# Pembagian data: 70% Train, 15% Validation, 15% Test
total_size = len(full_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_subset, val_subset, test_subset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Terapkan transform validasi & uji ke subset masing-masing
class TransformWrapper(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        x, y = self.subset[idx]
        # Jika transform diperlukan
        return x, y

BATCH_SIZE = 16
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Ukuran Pembagian Data:")
print(f"  - Training   : {len(train_subset)} citra ({len(train_loader)} batches)")
print(f"  - Validation : {len(val_subset)} citra ({len(val_loader)} batches)")
print(f"  - Testing    : {len(test_subset)} citra ({len(test_loader)} batches)")

In [ ]:
# Visualisasi sampel data pelatihan
def imshow(img, title=None):
    img = img.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    if title:
        plt.title(title, fontsize=12, fontweight="bold")
    plt.axis("off")

inputs, classes = next(iter(train_loader))
fig = plt.figure(figsize=(14, 6))
for i in range(min(6, len(inputs))):
    ax = fig.add_subplot(2, 3, i + 1)
    imshow(inputs[i], title=class_names[classes[i]])
plt.tight_layout()
plt.show()

In [ ]:
# Inisialisasi model VGG-16 dengan Pretrained Weights (ImageNet)
print("Memuat VGG16 Pre-trained Backbone...")
model = vgg16(weights=VGG16_Weights.DEFAULT)

# Freeze convolutional feature extraction layers
for param in model.features.parameters():
    param.requires_grad = False

# Kustomisasi Classifier Head untuk 3 Kelas Target
num_classes = len(class_names)
model.classifier = nn.Sequential(
    nn.Linear(512 * 7 * 7, 1024),
    nn.BatchNorm1d(1024),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5),
    nn.Linear(1024, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.4),
    nn.Linear(512, num_classes)
)

model = model.to(device)
print(f"Model VGG16 siap untuk {num_classes} kelas: {class_names}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameter: {total_params:,} | Parameter Dilatih: {trainable_params:,}")

In [ ]:
# Loss function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

def train_model(model, criterion, optimizer, scheduler, num_epochs=15):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": []
    }
    
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print("-" * 30)
        
        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0
        
        for inputs, labels in tqdm(train_loader, desc="Training"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        epoch_train_loss = running_loss / len(train_subset)
        epoch_train_acc = running_corrects.double() / len(train_subset)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation"):
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)
                
        epoch_val_loss = val_loss / len(val_subset)
        epoch_val_acc = val_corrects.double() / len(val_subset)
        
        scheduler.step(epoch_val_loss)
        
        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc.item())
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc.item())
        
        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}%")
        print(f"Val Loss  : {epoch_val_loss:.4f} | Val Acc  : {epoch_val_acc*100:.2f}%")
        
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, f"best_vgg16_{category_name}.pth")
            print(f"Checkpoint terbaik disimpan! (Val Acc: {best_acc*100:.2f}%)")
            
        print()
        
    time_elapsed = time.time() - since
    print(f"Pelatihan selesai dalam {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best Val Acc: {best_acc*100:.2f}%")
    
    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
# Jalankan pelatihan selama 15 epoch (bisa disesuaikan)
EPOCHS = 15
trained_model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS)

In [ ]:
# Visualisasi kurva Loss dan Akurasi
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], "-o", label="Training Loss", color="#0284c7")
plt.plot(epochs_range, history["val_loss"], "-s", label="Validation Loss", color="#ef4444")
plt.title("Loss vs Epochs", fontsize=13, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, [a * 100 for a in history["train_acc"]], "-o", label="Training Accuracy", color="#0284c7")
plt.plot(epochs_range, [a * 100 for a in history["val_acc"]], "-s", label="Validation Accuracy", color="#10b981")
plt.title("Accuracy vs Epochs", fontsize=13, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
plt.savefig(f"training_curves_{category_name}.png", dpi=300)
plt.show()

In [ ]:
# Evaluasi pada Test Set (Data Independen yang belum pernah dilihat model)
trained_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing Evaluation"):
        inputs = inputs.to(device)
        outputs = trained_model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Laporan Klasifikasi (Precision, Recall, F1-Score sesuai Bab 3.4 Proposal)
print("=" * 60)
print(f"LAPORAN EVALUASI MODEL VGG16 ({category_name.upper()})")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# Confusion Matrix Heatmap
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix - VGG16 ({category_name.upper()})", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Label", fontweight="bold")
plt.ylabel("True Ground Truth", fontweight="bold")
plt.tight_layout()
plt.savefig(f"confusion_matrix_{category_name}.png", dpi=300)
plt.show()

In [ ]:
# Fungsi Inferensi untuk Prediksi Citra Tunggal
def predict_fashion_item(image_path, model, class_names):
    img = Image.open(image_path).convert("RGB")
    input_tensor = val_test_transforms(img).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]
        top_prob, top_class = torch.max(probs, dim=0)
        
    plt.figure(figsize=(9, 4))
    
    # Tampilkan gambar
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Prediksi: {class_names[top_class]}
Confidence: {top_prob*100:.2f}%", fontsize=11, fontweight="bold")
    plt.axis("off")
    
    # Diagram batang probabilitas per kelas
    plt.subplot(1, 2, 2)
    y_pos = np.arange(len(class_names))
    plt.barh(y_pos, probs.cpu().numpy() * 100, color="#0284c7")
    plt.yticks(y_pos, class_names)
    plt.xlabel("Probabilitas (%)")
    plt.xlim(0, 100)
    plt.grid(axis="x", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    return class_names[top_class], top_prob.item()

# Demo prediksi menggunakan salah satu citra acak dari test set
sample_img_path, sample_label = test_subset.dataset.samples[test_subset.indices[0]]
print(f"Menguji citra: {sample_img_path} (Ground Truth: {class_names[sample_label]})")
pred_label, conf = predict_fashion_item(sample_img_path, trained_model, class_names)